# one ship near the shoreline

In [ ]:
from pathlib import Path
import hashlib, importlib, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as PolygonPatch
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from netCDF4 import Dataset
from shapely import wkt
from shapely.geometry import box
from shapely.ops import unary_union
sys.path.insert(0, str(Path('../../src').resolve()))
import geoclaw_debris.ship_tracking as ship_tracking
importlib.reload(ship_tracking)
from geoclaw_debris.ship_tracking import Flow, Ship, simulate, hull_polygon

ds = Dataset('pike_place_buffered_fgout.nc')
assert ds.terrain_sha256 == hashlib.sha256(Path('seattle_buildings_mhw.tt3').read_bytes()).hexdigest()
flow = Flow(ds, current_scale=5.)
extent = box(flow.lon.min(), flow.lat.min(), flow.lon.max(), flow.lat.max())
buildings = unary_union([p for p in (wkt.loads(s) for s in pd.read_csv('../../SeattleBuildings/Seattle_Inventory.csv').geometry) if p.intersects(extent)])

In [ ]:
ship = Ship()
start = (-122.342, 47.605)
track = pd.DataFrame(simulate(flow, buildings, ship, start))
net = np.hypot(track.x_m.iloc[-1]-track.x_m.iloc[0], track.y_m.iloc[-1]-track.y_m.iloc[0])
print(f'net drift: {net:.1f} m; final state: {track.status.iloc[-1]}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
origin = track[['x_m','y_m']].iloc[0].to_numpy()
ix = abs(flow.lon-start[0]) < .003
iy = abs(flow.lat-start[1]) < .003
lon, lat = np.meshgrid(flow.lon[ix], flow.lat[iy], indexing='ij')
x, y = flow.to_xy(lon, lat)
depth = np.asarray(ds['h'][12])[np.ix_(ix, iy)]
ax.pcolormesh(x-origin[0], y-origin[1], np.ma.masked_where(depth < .1, depth), cmap='Blues', vmin=0, vmax=30, shading='auto')
row = track.iloc[0]
outline = hull_polygon((row.x_m,row.y_m), 0., ship)
hull = PolygonPatch(np.asarray(outline.exterior.coords)-origin, facecolor='tomato', edgecolor='red', lw=2, alpha=.8)
ax.add_patch(hull)
trail, = ax.plot([], [], 'r-', lw=2)
ax.set(xlim=(-140,210), ylim=(-130,130), xlabel='east (m)', ylabel='north (m)')
ax.set_aspect('equal')
ax.set_title('ship near shoreline: 0 s, afloat')
fig.tight_layout()

def update(i):
    row = track.iloc[i]
    outline = hull_polygon((row.x_m,row.y_m), np.deg2rad(row.heading_deg), ship)
    hull.set_xy(np.asarray(outline.exterior.coords)-origin)
    trail.set_data(track.x_m.iloc[:i+1]-origin[0], track.y_m.iloc[:i+1]-origin[1])
    ax.set_title(f'ship near shoreline: {int(row.time_s)} s, {row.status}')

anim = FuncAnimation(fig, update, frames=range(0,601,10), interval=120)
plt.close(fig)
HTML(anim.to_jshtml())